
---

# FPO 论文学习笔记

**论文：** Flow Matching Policy Gradients  
**方法：** Flow Policy Optimization（FPO）  
**论文地址：** https://arxiv.org/pdf/2507.21053

这版笔记按照以下顺序组织：

1. 研究背景与核心问题
2. Flow Matching Policy 与 PPO 的矛盾
3. ELBO 的基本数学原理
4. **从 DDPM / VPSDE 严格推导 ELBO**
5. **从 Flow Matching 推导 Expected ELBO**
6. FPO Ratio 的数学推导
7. Monte Carlo 估计及偏差
8. FPO 的 PPO 优化目标
9. 完整训练流程
10. 实验设计与结果
11. FPO、ReinFlow 与 Diffusion-GRPO 的区别
12. 理论局限与适用范围
13. 总结

其中第 3～6 节是整篇论文最重要的理论部分。

---

这里首先要明确一个关键区别：

**VPSDE 扩散模型与 Flow Matching 都可以建立 ELBO 联系，但不能直接证明普通 Flow Matching Loss 等于干净数据的负 ELBO。**

更严谨的推导需要分成三个层次：

1. **VPSDE：** 从离散扩散模型的变分下界出发，推导出加权噪声预测 MSE，并说明连续时间极限。
2. **Flow Matching：** 将 OT 直线路径转换为高斯加噪路径，推导速度预测与噪声预测的关系，再通过 Log-SNR 权重连接到扩散模型的变分目标。
3. **FPO：** 区分干净动作的 ELBO、噪声增强动作的 Expected ELBO，以及实际使用的 Monte Carlo FM Loss Ratio，明确哪些是严格等式，哪些只是代理关系。

尤其是第三点：**即使两个模型的 ELBO 都是各自对数概率的下界，也不能直接认为两个 ELBO 的差就是两个真实对数概率的差。** 这是 FPO 理论中必须保留的限定。

---

# 一、研究背景与核心问题

传统连续动作强化学习通常使用高斯策略：

$$
\pi_\theta(a|s)
=
\mathcal N(a;\mu_\theta(s),\Sigma_\theta(s))
$$

其中：

- $s$：环境状态。
- $a$：动作。
- $\theta$：策略网络参数。
- $\mu_\theta(s)$：动作均值。
- $\Sigma_\theta(s)$：动作协方差。

高斯策略的优势是可以直接计算：

$$
\log\pi_\theta(a|s)
$$

因此容易使用 PPO 等策略梯度算法。

但单高斯策略难以表达复杂的多模态动作分布。

例如，在同一个环境状态下，机器人可能存在两种合理行为：

$$
a_{\rm left},\qquad a_{\rm right}
$$

理想策略可能具有两个概率峰值。

Flow Matching 可以通过 ODE 将简单高斯噪声转换成复杂动作分布：

$$
z\sim\mathcal N(0,I)
$$

$$
\frac{dx_\tau}{d\tau}
=
v_\theta(x_\tau,\tau,s)
$$

$$
a=x_1
$$

其中：

- $\tau\in[0,1]$：Flow 生成时间。
- $x_0=z$：初始高斯噪声。
- $x_1=a$：最终动作。
- $v_\theta$：速度场网络。

因此，Flow Policy 可以表达更复杂的动作分布。

但问题是：

**Flow Matching 容易采样，却不容易高效计算生成动作的概率密度。**

这正是 FPO 要解决的问题。[arXiv](https://arxiv.org/html/2507.21053?utm_source=chatgpt.com)

---

# 二、为什么 Flow Matching 难以直接使用 PPO？

PPO 的核心目标为：

$$
J_{\rm PPO}
=
\mathbb E
\left[
\min
\left(
r_\theta A,
\operatorname{clip}(r_\theta,1-\epsilon,1+\epsilon)A
\right)
\right]
$$

其中：

$$
\boxed{
r_\theta
=
\frac{\pi_\theta(a|s)}
{\pi_{\rm old}(a|s)}
}
$$

因此，需要计算当前策略和旧策略生成同一个动作的概率。

对于 Flow ODE：

$$
\frac{dx_\tau}{d\tau}=v_\theta(x_\tau,\tau,s)
$$

理论上可以通过连续变量替换公式计算：

$$
\log\pi_\theta(a|s)
=
\log p_0(x_0)
-
\int_0^1
\nabla\cdot v_\theta(x_\tau,\tau,s)\,d\tau
$$

其中：

$$
\nabla\cdot v_\theta
=
\operatorname{Tr}
\left(
\frac{\partial v_\theta}{\partial x_\tau}
\right)
$$

但这种方式需要额外计算速度场散度，并求解相应的 ODE。

对于高维动作空间，计算成本较高。

因此，FPO 不直接计算真实动作概率，而是使用：

$$
\boxed{
\operatorname{ELBO}_\theta(a|s)
}
$$

作为对数概率的代理。

进一步利用 Flow Matching Loss 构造：

$$
\boxed{
r_{\rm FPO}
=
\exp
\left(
L_{\rm old}-L_\theta
\right)
}
$$

下面开始推导这一公式。

---

# 三、ELBO 的基本数学原理

首先需要明确：

**ELBO 并不是扩散模型特有的概念，而是变分推断中的通用结果。**

## 3.1 引入隐变量

假设生成模型包含观测变量 $x$ 和隐变量 $z$。

联合分布为：

$$
p_\theta(x,z)
=
p_\theta(x|z)p(z)
$$

我们希望最大化数据的对数概率：

$$
\log p_\theta(x)
$$

其中：

$$
p_\theta(x)
=
\int p_\theta(x,z)\,dz
$$

由于积分可能难以计算，引入一个辅助分布：

$$
q(z|x)
$$

于是：

$$
\begin{aligned}
\log p_\theta(x)
&=
\log\int p_\theta(x,z)\,dz\\
&=
\log\int
q(z|x)
\frac{p_\theta(x,z)}{q(z|x)}
\,dz\\
&=
\log
\mathbb E_{q(z|x)}
\left[
\frac{p_\theta(x,z)}{q(z|x)}
\right]
\end{aligned}
$$

根据 Jensen 不等式：

$$
\log\mathbb E[X]
\geq
\mathbb E[\log X]
$$

得到：

$$
\boxed{
\log p_\theta(x)
\geq
\mathbb E_{q(z|x)}
\left[
\log p_\theta(x,z)-\log q(z|x)
\right]
}
$$

定义：

$$
\boxed{
\operatorname{ELBO}_\theta(x)
=
\mathbb E_q
\left[
\log p_\theta(x,z)-\log q(z|x)
\right]
}
$$

因此：

$$
\boxed{
\operatorname{ELBO}_\theta(x)
\leq
\log p_\theta(x)
}
$$

## 3.2 ELBO 与真实概率之间相差什么？

根据贝叶斯公式：

$$
p_\theta(z|x)
=
\frac{p_\theta(x,z)}{p_\theta(x)}
$$

计算：

$$
\begin{aligned}
D_{\rm KL}(q(z|x)\|p_\theta(z|x))
&=
\mathbb E_q
\left[
\log\frac{q(z|x)}{p_\theta(z|x)}
\right]\\
&=
\mathbb E_q
\left[
\log q(z|x)-\log p_\theta(x,z)
\right]
+\log p_\theta(x)
\end{aligned}
$$

整理得到：

$$
\boxed{
\log p_\theta(x)
=
\operatorname{ELBO}_\theta(x)
+
D_{\rm KL}(q(z|x)\|p_\theta(z|x))
}
$$

这就是 ELBO 最重要的数学关系。

定义：

$$
D_\theta(x)
=
D_{\rm KL}(q(z|x)\|p_\theta(z|x))
$$

则：

$$
\boxed{
\operatorname{ELBO}_\theta(x)
=
\log p_\theta(x)-D_\theta(x)
}
$$

因此：

- 最大化 ELBO 可以提高一个对数概率下界。
- 当 KL Gap 为零时，ELBO 等于真实对数概率。
- 当 KL Gap 不为零时，ELBO 只是代理。

接下来需要证明：

**为什么扩散模型中的噪声预测 MSE 能够对应这个 ELBO？**

---

# 四、从 DDPM / VPSDE 推导 ELBO

这一节从离散扩散模型开始，再推导到连续时间 VPSDE。

这样可以清楚看到：

$$
\boxed{
\text{ELBO}
\rightarrow
\text{反向高斯 KL}
\rightarrow
\text{噪声预测 MSE}
}
$$

## 4.1 DDPM 的前向过程

定义干净数据：

$$
x_0\sim p_{\rm data}(x)
$$

前向扩散：

$$
\boxed{
q(x_k|x_{k-1})
=
\mathcal N
\left(
\sqrt{1-\beta_k}x_{k-1},
\beta_k I
\right)
}
$$

其中：

- $k=1,\ldots,K$：离散扩散时间。
- $\beta_k$：第 $k$ 步噪声方差。

定义：

$$
\alpha_k=1-\beta_k
$$

$$
\bar\alpha_k=\prod_{j=1}^k\alpha_j
$$

根据高斯分布的递推性质，可以得到：

$$
\boxed{
q(x_k|x_0)
=
\mathcal N
\left(
\sqrt{\bar\alpha_k}x_0,
(1-\bar\alpha_k)I
\right)
}
$$

因此可以直接采样：

$$
\boxed{
x_k
=
\sqrt{\bar\alpha_k}x_0
+
\sqrt{1-\bar\alpha_k}\epsilon
}
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

这就是 DDPM 训练时能够直接随机采样任意扩散时间的原因。

## 4.2 构造反向生成模型

反向模型定义为：

$$
p_\theta(x_{0:K})
=
p(x_K)
\prod_{k=1}^{K}
p_\theta(x_{k-1}|x_k)
$$

其中：

$$
p(x_K)=\mathcal N(0,I)
$$

我们希望最大化：

$$
\log p_\theta(x_0)
$$

但需要对所有中间隐变量积分：

$$
p_\theta(x_0)
=
\int
p_\theta(x_{0:K})\,dx_{1:K}
$$

因此引入前向分布：

$$
q(x_{1:K}|x_0)
$$

构造 ELBO：

$$
\begin{aligned}
\log p_\theta(x_0)
&\geq
\mathbb E_q
\left[
\log p_\theta(x_{0:K})
-
\log q(x_{1:K}|x_0)
\right]
\end{aligned}
$$

定义：

$$
\boxed{
\operatorname{ELBO}_\theta(x_0)
=
\mathbb E_q
\left[
\log p_\theta(x_{0:K})
-
\log q(x_{1:K}|x_0)
\right]
}
$$

这一步是严格的 Jensen 下界。

## 4.3 将 ELBO 分解成 KL 散度

利用前向过程的马尔可夫性质：

$$
q(x_{1:K}|x_0)
=
\prod_{k=1}^{K}q(x_k|x_{k-1})
$$

可以将负 ELBO 分解为：

$$
\boxed{
\begin{aligned}
-\operatorname{ELBO}_\theta(x_0)
={}&
D_{\rm KL}
\left(
q(x_K|x_0)\|p(x_K)
\right)\\
&+
\sum_{k=2}^{K}
\mathbb E_{q(x_k|x_0)}
\left[
D_{\rm KL}
\left(
q(x_{k-1}|x_k,x_0)
\|
p_\theta(x_{k-1}|x_k)
\right)
\right]\\
&-
\mathbb E_q
\left[
\log p_\theta(x_0|x_1)
\right]
\end{aligned}
}
$$

三个部分分别是：

**第一项：先验匹配。**

要求最终前向分布接近：

$$
\mathcal N(0,I)
$$

**第二项：反向过程匹配。**

要求模型预测的反向高斯转移接近真实后验。

**第三项：重建项。**

要求最后一步能够恢复干净数据。

我们重点关注第二项，因为它将产生噪声预测 MSE。

## 4.4 推导真实反向后验

需要计算：

$$
q(x_{k-1}|x_k,x_0)
$$

根据贝叶斯公式：

$$
q(x_{k-1}|x_k,x_0)
\propto
q(x_k|x_{k-1})
q(x_{k-1}|x_0)
$$

其中：

$$
q(x_k|x_{k-1})
=
\mathcal N(\sqrt{\alpha_k}x_{k-1},\beta_k I)
$$

$$
q(x_{k-1}|x_0)
=
\mathcal N
\left(
\sqrt{\bar\alpha_{k-1}}x_0,
(1-\bar\alpha_{k-1})I
\right)
$$

两个关于 $x_{k-1}$ 的高斯密度相乘，仍然得到高斯分布：

$$
\boxed{
q(x_{k-1}|x_k,x_0)
=
\mathcal N
\left(
\widetilde\mu_k(x_k,x_0),
\widetilde\beta_k I
\right)
}
$$

其中：

$$
\boxed{
\widetilde\beta_k
=
\frac{1-\bar\alpha_{k-1}}
{1-\bar\alpha_k}\beta_k
}
$$

后验均值：

$$
\boxed{
\widetilde\mu_k
=
\frac{
\sqrt{\bar\alpha_{k-1}}\beta_k
}{
1-\bar\alpha_k
}x_0
+
\frac{
\sqrt{\alpha_k}(1-\bar\alpha_{k-1})
}{
1-\bar\alpha_k
}x_k
}
$$

利用：

$$
x_k
=
\sqrt{\bar\alpha_k}x_0
+
\sqrt{1-\bar\alpha_k}\epsilon
$$

消去 $x_0$，可以得到更常用的形式：

$$
\boxed{
\widetilde\mu_k
=
\frac1{\sqrt{\alpha_k}}
\left(
x_k-
\frac{\beta_k}{\sqrt{1-\bar\alpha_k}}\epsilon
\right)
}
$$

注意：这个后验依赖真实 $x_0$，因此可以在训练阶段计算，但生成阶段无法直接使用。

## 4.5 用神经网络预测反向均值

定义模型反向分布：

$$
p_\theta(x_{k-1}|x_k)
=
\mathcal N
\left(
\mu_\theta(x_k,k),
\sigma_k^2I
\right)
$$

使用噪声预测网络：

$$
\epsilon_\theta(x_k,k)
$$

参数化均值：

$$
\boxed{
\mu_\theta(x_k,k)
=
\frac1{\sqrt{\alpha_k}}
\left(
x_k-
\frac{\beta_k}{\sqrt{1-\bar\alpha_k}}
\epsilon_\theta(x_k,k)
\right)
}
$$

于是：

$$
\widetilde\mu_k-\mu_\theta
=
\frac{\beta_k}
{\sqrt{\alpha_k}\sqrt{1-\bar\alpha_k}}
(\epsilon_\theta-\epsilon)
$$

现在计算反向高斯之间的 KL 散度。

对于相同的固定协方差：

$$
D_{\rm KL}
\left(
\mathcal N(\mu_1,\sigma^2I)
\|
\mathcal N(\mu_2,\sigma^2I)
\right)
=
\frac1{2\sigma^2}
\|\mu_1-\mu_2\|^2
$$

如果模型协方差和真实后验协方差不完全相同，只要二者均固定、不依赖 $\theta$，与均值网络相关的部分仍具有相同形式，其余项可以并入常数。

因此：

$$
\boxed{
D_{\rm KL}
\left(
q(x_{k-1}|x_k,x_0)
\|
p_\theta(x_{k-1}|x_k)
\right)
=
\frac{\beta_k^2}
{2\sigma_k^2\alpha_k(1-\bar\alpha_k)}
\|\epsilon_\theta-\epsilon\|^2
+C_k
}
$$

其中 $C_k$ 不依赖均值网络参数。

于是：

$$
\boxed{
-\operatorname{ELBO}_\theta(x_0)
=
\sum_{k=2}^{K}
w_k
\mathbb E_\epsilon
\left[
\|\epsilon_\theta(x_k,k)-\epsilon\|^2
\right]
+
L_{\rm prior}
+
L_{\rm recon}
+
C
}
$$

其中：

$$
\boxed{
w_k
=
\frac{\beta_k^2}
{2\sigma_k^2\alpha_k(1-\bar\alpha_k)}
}
$$

这就建立了：

$$
\boxed{
\text{DDPM 反向高斯 KL}
\longrightarrow
\text{加权噪声预测 MSE}
}
$$

需要注意：**DDPM 实践中经常使用的无权重噪声 MSE，是对原始变分目标的重新加权，并不一般等于严格负 ELBO。**

---

## 4.6 从 DDPM 过渡到 VPSDE

现在考虑连续时间前向 SDE：

$$
\boxed{
dx_t
=
-\frac12\beta(t)x_t\,dt
+
\sqrt{\beta(t)}\,dW_t
}
$$

其中：

- $t\in[0,T]$：连续扩散时间。
- $\beta(t)>0$：噪声调度。
- $W_t$：标准布朗运动。

这是 Variance Preserving SDE，简称 VPSDE。

定义：

$$
\alpha(t)
=
\exp
\left(
-\frac12\int_0^t\beta(u)\,du
\right)
$$

$$
\sigma^2(t)=1-\alpha^2(t)
$$

其条件边缘分布为：

$$
\boxed{
q(x_t|x_0)
=
\mathcal N
\left(
\alpha(t)x_0,
\sigma^2(t)I
\right)
}
$$

因此：

$$
\boxed{
x_t=\alpha(t)x_0+\sigma(t)\epsilon
}
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

与离散 DDPM 相比，只是将：

$$
\sqrt{\bar\alpha_k}
$$

替换成：

$$
\alpha(t)
$$

将：

$$
\sqrt{1-\bar\alpha_k}
$$

替换成：

$$
\sigma(t)
$$

---

## 4.7 连续时间下的 ELBO 权重

定义 Log-SNR：

$$
\boxed{
\lambda(t)
=
\log\frac{\alpha^2(t)}{\sigma^2(t)}
}
$$

由于：

$$
\alpha^2(t)
=
\exp
\left(
-\int_0^t\beta(u)\,du
\right)
$$

所以：

$$
\frac{d\alpha^2}{dt}
=
-\beta(t)\alpha^2(t)
$$

又因为：

$$
\sigma^2(t)=1-\alpha^2(t)
$$

因此：

$$
\frac{d\sigma^2}{dt}
=
\beta(t)\alpha^2(t)
$$

对 Log-SNR 求导：

$$
\begin{aligned}
\frac{d\lambda}{dt}
&=
\frac{d}{dt}
\left(
\log\alpha^2-\log\sigma^2
\right)\\
&=
-\beta(t)
-
\frac{\beta(t)\alpha^2(t)}{\sigma^2(t)}
\end{aligned}
$$

利用：

$$
\alpha^2+\sigma^2=1
$$

得到：

$$
\boxed{
-\frac{d\lambda}{dt}
=
\frac{\beta(t)}{\sigma^2(t)}
}
$$

现在考虑连续时间极限。

离散反向 KL 的均值误差权重在小步长极限下趋向：

$$
\frac{\beta(t)}{2\sigma^2(t)}\,dt
$$

因此：

$$
\boxed{
-\operatorname{ELBO}_\theta(x_0)
=
\frac12
\int_0^T
\frac{\beta(t)}{\sigma^2(t)}
\mathbb E_\epsilon
\left[
\|\epsilon_\theta(x_t,t)-\epsilon\|^2
\right]dt
+
C
}
$$

利用刚才的 Log-SNR 关系：

$$
\boxed{
-\operatorname{ELBO}_\theta(x_0)
=
\frac12
\int_0^T
\left(
-\frac{d\lambda}{dt}
\right)
\mathbb E_\epsilon
\left[
\|\epsilon_\theta(x_t,t)-\epsilon\|^2
\right]dt
+
C
}
$$

这里的 $C$ 包含与模型参数无关的边界项，并假设采用匹配的连续时间扩散变分模型、固定扩散系数和相应端点处理。

在这些条件下，这就是连续时间扩散模型负 ELBO 的标准形式。[Hugging Face](https://huggingface.co/buckets/huggingchat/papers-content/tree/2303/2303.00848.md?utm_source=chatgpt.com)

如果令：

$$
t\sim U(0,T)
$$

则：

$$
\boxed{
-\operatorname{ELBO}_\theta(x_0)
=
\frac T2
\mathbb E_{t,\epsilon}
\left[
-\frac{d\lambda}{dt}
\|\epsilon_\theta(x_t,t)-\epsilon\|^2
\right]
+
C
}
$$

当 $T=1$ 时，前面的 $T$ 可以省略。

### 这一节的核心结论

对于 VPSDE：

$$
\boxed{
\begin{aligned}
&\text{前向高斯过程}\\
&\Downarrow\\
&\text{反向过程的变分 ELBO}\\
&\Downarrow\\
&\text{每一步反向高斯 KL}\\
&\Downarrow\\
&\text{加权噪声预测 MSE}\\
&\Downarrow\\
&-\operatorname{ELBO}_\theta
=
\frac12\mathbb E
\left[
-\dot\lambda
\|\epsilon_\theta-\epsilon\|^2
\right]+C
\end{aligned}
}
$$

**因此，VPSDE 中用加权去噪 MSE 计算 ELBO 不是经验假设，而是可以从扩散变分模型推导出来的。**

---

# 五、从 Flow Matching 推导 Expected ELBO

这是 FPO 最关键的理论扩展。

我们已经知道：

$$
\boxed{
-\operatorname{ELBO}
=
\frac12
\mathbb E
\left[
-\dot\lambda
\|\epsilon_\theta-\epsilon\|^2
\right]+C
}
$$

但 Flow Matching 训练的是速度场：

$$
v_\theta
$$

而不是直接预测噪声。

接下来需要证明：

1. Flow Matching 的速度 MSE 可以转换成噪声 MSE。
2. 转换后的目标属于加权扩散损失。
3. 特定权重满足 Kingma–Gao 的 Expected ELBO 定理。

## 5.1 定义 OT-CFM 路径

为了与前面 VPSDE 的数据到噪声方向保持一致，这里仍然使用：

- $x_0$：干净数据。
- $t=0$：数据端。
- $t=1$：噪声端。

定义 OT 直线路径：

$$
\boxed{
x_t=(1-t)x_0+t\epsilon
}
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

因此：

$$
\alpha(t)=1-t
$$

$$
\sigma(t)=t
$$

条件目标速度为：

$$
\boxed{
u_t=\epsilon-x_0
}
$$

标准 OT-CFM Loss：

$$
\boxed{
L_{\rm FM}(\theta;x_0)
=
\mathbb E_{t,\epsilon}
\left[
\|v_\theta(x_t,t)-(\epsilon-x_0)\|^2
\right]
}
$$

注意，这里使用的是**数据到噪声方向**，所以目标速度为：

$$
\epsilon-x_0
$$

如果使用噪声到数据方向，目标速度则为：

$$
x_0-\epsilon
$$

两种写法可以通过时间反转互相转换。

## 5.2 从速度预测转换为噪声预测

由：

$$
x_t=(1-t)x_0+t\epsilon
$$

以及：

$$
u_t=\epsilon-x_0
$$

可以得到：

$$
x_0=x_t-tu_t
$$

进一步：

$$
\epsilon=x_t+(1-t)u_t
$$

因此模型的噪声预测可以写成：

$$
\boxed{
\epsilon_\theta(x_t,t)
=
x_t+(1-t)v_\theta(x_t,t)
}
$$

于是：

$$
\begin{aligned}
\epsilon_\theta-\epsilon
&=
x_t+(1-t)v_\theta
-
[x_t+(1-t)u_t]\\
&=
(1-t)(v_\theta-u_t)
\end{aligned}
$$

所以：

$$
\boxed{
\|\epsilon_\theta-\epsilon\|^2
=
(1-t)^2
\|v_\theta-u_t\|^2
}
$$

这是 Flow Matching 与扩散噪声预测之间的第一个关键关系。

## 5.3 计算 OT 路径的 Log-SNR

因为：

$$
\alpha(t)=1-t
$$

$$
\sigma(t)=t
$$

所以：

$$
\lambda(t)
=
\log\frac{(1-t)^2}{t^2}
$$

即：

$$
\boxed{
\lambda(t)
=
2\log\frac{1-t}{t}
}
$$

求导：

$$
\frac{d\lambda}{dt}
=
-\frac2{1-t}-\frac2t
$$

整理：

$$
\boxed{
-\frac{d\lambda}{dt}
=
\frac2{t(1-t)}
}
$$

## 5.4 代入标准负 ELBO 公式

回顾：

$$
L_{\rm ELBO}
=
\frac12
\mathbb E
\left[
-\frac{d\lambda}{dt}
\|\epsilon_\theta-\epsilon\|^2
\right]
$$

代入：

$$
-\frac{d\lambda}{dt}
=
\frac2{t(1-t)}
$$

以及：

$$
\|\epsilon_\theta-\epsilon\|^2
=
(1-t)^2\|v_\theta-u_t\|^2
$$

得到：

$$
\begin{aligned}
L_{\rm ELBO}
&=
\frac12
\mathbb E
\left[
\frac2{t(1-t)}
(1-t)^2
\|v_\theta-u_t\|^2
\right]\\
&=
\mathbb E
\left[
\frac{1-t}{t}
\|v_\theta-u_t\|^2
\right]
\end{aligned}
$$

因此：

$$
\boxed{
L_{\rm ELBO}
=
\mathbb E
\left[
\frac{1-t}{t}
\|v_\theta-u_t\|^2
\right]
}
$$

这里忽略了与模型参数无关的边界常数。

这个公式说明：

**如果希望 OT 路径对应标准的干净数据 ELBO，那么速度 MSE 需要乘以时间权重：**

$$
\boxed{
\frac{1-t}{t}
}
$$

而普通 Flow Matching 使用的是：

$$
\boxed{
L_{\rm FM}
=
\mathbb E
\left[
\|v_\theta-u_t\|^2
\right]
}
$$

因此：

$$
\boxed{
L_{\rm FM}\neq L_{\rm ELBO}
}
$$

这是必须明确区分的。

---

## 5.5 普通 Flow Matching 对应什么加权扩散损失？

定义一般加权去噪目标：

$$
\boxed{
L_w
=
\frac12
\mathbb E
\left[
w(\lambda(t))
\left(
-\frac{d\lambda}{dt}
\right)
\|\epsilon_\theta-\epsilon\|^2
\right]
}
$$

代入前面的关系：

$$
L_w
=
\mathbb E
\left[
w(\lambda(t))
\frac{1-t}{t}
\|v_\theta-u_t\|^2
\right]
$$

如果希望：

$$
L_w=L_{\rm FM}
$$

则需要：

$$
w(\lambda(t))
\frac{1-t}{t}=1
$$

因此：

$$
\boxed{
w(\lambda(t))
=
\frac{t}{1-t}
}
$$

因为：

$$
e^{\lambda(t)/2}
=
\frac{1-t}{t}
$$

所以：

$$
\boxed{
w(\lambda)=e^{-\lambda/2}
}
$$

最终：

$$
\boxed{
L_{\rm FM}
=
L_w
\quad\text{当}\quad
w(\lambda)=e^{-\lambda/2}
}
$$

这个结果非常重要。

它说明：

**OT-CFM 的普通速度 MSE，是加权扩散目标的一个特例。**

但还没有证明它等于某个 ELBO。

下一步需要引入 Kingma–Gao 的定理。

---

## 5.6 Kingma–Gao：加权扩散损失与 Expected ELBO

Kingma 和 Gao 在 NeurIPS 2023 论文：

[Understanding Diffusion Objectives as the ELBO with Simple Data Augmentation](https://arxiv.org/abs/2303.00848)

中证明：

**当加权函数满足单调性条件时，加权扩散目标可以解释为不同噪声水平下的 ELBO 期望。** [NeurIPS Proceedings](https://proceedings.neurips.cc/paper_files/paper/2023/hash/ce79fbf9baef726645bc2337abb0ade2-Abstract-Conference.html?utm_source=chatgpt.com)

下面推导这个结论。

### 第一步：定义截断扩散过程的变分目标

对于干净数据 $x_0$，考虑从时间 $t$ 开始的扩散过程。

定义：

$$
\boxed{
\mathcal K_\theta(t;x_0)
=
D_{\rm KL}
\left(
q(x_{t:T}|x_0)
\|
p_\theta(x_{t:T})
\right)
}
$$

这里采用连续时间记号，严格定义可通过离散时间极限理解。

$\mathcal K_\theta(t;x_0)$ 表示从噪声水平 $t$ 开始，前向过程与反向生成模型之间的 KL 散度。

它与该噪声水平下的负 ELBO 相差一个与模型参数无关的项。

Kingma–Gao 给出其时间导数：

$$
\boxed{
\frac{d\mathcal K_\theta(t;x_0)}{dt}
=
\frac12
\frac{d\lambda(t)}{dt}
\mathbb E_\epsilon
\left[
\|\epsilon_\theta-\epsilon\|^2
\right]
}
$$

因为：

$$
\frac{d\lambda}{dt}<0
$$

所以：

$$
\frac{d\mathcal K_\theta}{dt}\leq0
$$

直观上，截断掉更多低噪声阶段后，需要匹配的反向过程减少，KL 散度也会减小。

### 第二步：把加权去噪损失写成 KL 导数

加权扩散损失：

$$
L_w
=
\frac12
\int_0^T
w(t)
\left(
-\frac{d\lambda}{dt}
\right)
\mathbb E_\epsilon
\left[
\|\epsilon_\theta-\epsilon\|^2
\right]dt
$$

这里简写：

$$
w(t)=w(\lambda(t))
$$

利用前面的导数关系：

$$
\boxed{
L_w
=
-\int_0^T
w(t)
\frac{d\mathcal K_\theta(t;x_0)}{dt}
\,dt
}
$$

### 第三步：分部积分

根据：

$$
\int u\,dv=uv-\int v\,du
$$

得到：

$$
\begin{aligned}
L_w
&=
-\left[
w(t)\mathcal K_\theta(t;x_0)
\right]_0^T\\
&\quad+
\int_0^T
w'(t)\mathcal K_\theta(t;x_0)\,dt
\end{aligned}
$$

即：

$$
\boxed{
L_w
=
w(0)\mathcal K_\theta(0;x_0)
-
w(T)\mathcal K_\theta(T;x_0)
+
\int_0^T
w'(t)\mathcal K_\theta(t;x_0)\,dt
}
$$

这就是最重要的一步。

它把原来的**加权去噪 MSE**，转换成了**不同噪声水平下 KL 散度的加权积分**。

### 第四步：利用单调性构造概率分布

如果：

$$
w'(t)\geq0
$$

那么：

$$
w'(t)
$$

是非负的。

在适当的归一化和端点处理下，可以定义噪声时间分布：

$$
p_w(t)\propto w'(t)
$$

同时，若 $w(0)>0$，还需要保留位于 $t=0$ 的概率质量。

于是：

$$
\boxed{
L_w
=
C_w
\mathbb E_{t\sim p_w}
\left[
\mathcal K_\theta(t;x_0)
\right]
+
C
}
$$

其中：

- $C_w>0$：由权重归一化决定的常数。
- $C$：与模型参数无关的边界项。

当权重按照论文的规范归一化时，可以令相应系数为 1。

由于：

$$
\mathcal K_\theta(t;x_0)
=
-\mathbb E_{q(x_t|x_0)}
\left[
\operatorname{ELBO}_{\theta,t}(x_t)
\right]
+
C_t
$$

所以：

$$
\boxed{
L_w
=
-
C_w
\mathbb E_{t\sim p_w}
\mathbb E_{x_t\sim q(x_t|x_0)}
\left[
\operatorname{ELBO}_{\theta,t}(x_t)
\right]
+
C'
}
$$

这就是：

$$
\boxed{
\text{Weighted Diffusion Loss}
\longleftrightarrow
\text{Expected ELBO}
}
$$

需要特别注意：这里的 ELBO 对象是**噪声增强后的数据 $x_t$**，不是始终针对原始干净数据 $x_0$。

---

## 5.7 为什么 OT-CFM 满足这个定理？

前面推导得到：

$$
w(\lambda(t))=\frac{t}{1-t}
$$

求导：

$$
\boxed{
w'(t)=\frac1{(1-t)^2}>0
}
$$

因此，OT-CFM 的权重满足所需的单调性。

所以普通 OT-CFM Loss 可以通过 Kingma–Gao 的结果，获得 Expected ELBO 解释。

形式上：

$$
\boxed{
L_{\rm FM}(\theta;x_0)
=
-
C_w
\mathbb E_{t\sim p_w}
\mathbb E_{x_t\sim q(x_t|x_0)}
\left[
\operatorname{ELBO}_{\theta,t}(x_t)
\right]
+
C
}
$$

但这里还有一个需要严谨处理的问题。

对于理想 OT 路径：

$$
w(t)=\frac{t}{1-t}
$$

当：

$$
t\rightarrow1
$$

时：

$$
w(t)\rightarrow\infty
$$

因此，不能在完整开区间上直接把：

$$
w'(t)=\frac1{(1-t)^2}
$$

归一化为普通概率密度。

严格应用上述概率分布解释时，需要采用有限 Log-SNR 端点、截断噪声区间或适当的极限处理。

例如，限制：

$$
t\in[t_{\min},t_{\max}]
$$

其中：

$$
0<t_{\min}<t_{\max}<1
$$

那么：

$$
w(t_{\max})<\infty
$$

可以定义归一化权重。

**因此，OT-CFM 的 Expected ELBO 解释成立需要相应的端点和归一化条件，不能忽略无限噪声端点直接写成无条件等式。**

FPO 在附录 A.1.1 和 A.1.2 中讨论了权重选择及 Expected ELBO 的联系。[arXiv](https://arxiv.org/html/2507.21053?utm_source=chatgpt.com)

---

# 六、从 ELBO 推导 FPO Ratio

现在已经有两条理论路径。

### VPSDE

$$
\boxed{
\text{加权噪声 MSE}
=
-\operatorname{ELBO}_{\rm clean}+C
}
$$

其中采用标准 ELBO 权重：

$$
w(\lambda)=1
$$

### OT-CFM

$$
\boxed{
\text{普通 FM MSE}
\longleftrightarrow
-\operatorname{ExpectedELBO}_{\rm noisy}
}
$$

其中：

$$
w(\lambda)=e^{-\lambda/2}
$$

现在可以推导 FPO Ratio。

## 6.1 从真实 PPO Ratio 开始

PPO 需要：

$$
\boxed{
r_{\rm PPO}
=
\frac{\pi_\theta(a|s)}
{\pi_{\rm old}(a|s)}
}
$$

取对数：

$$
\log r_{\rm PPO}
=
\log\pi_\theta(a|s)
-
\log\pi_{\rm old}(a|s)
$$

但 Flow Policy 的真实 likelihood 不容易计算。

因此考虑 ELBO：

$$
\operatorname{ELBO}_\theta(a|s)
=
\log\pi_\theta(a|s)-D_\theta(a,s)
$$

构造：

$$
\boxed{
r_{\rm ELBO}
=
\exp
\left(
\operatorname{ELBO}_\theta
-
\operatorname{ELBO}_{\rm old}
\right)
}
$$

代入：

$$
\begin{aligned}
r_{\rm ELBO}
&=
\exp
\left(
\log\pi_\theta-D_\theta
-\log\pi_{\rm old}+D_{\rm old}
\right)\\
&=
\frac{\pi_\theta}{\pi_{\rm old}}
\exp(D_{\rm old}-D_\theta)
\end{aligned}
$$

因此：

$$
\boxed{
r_{\rm ELBO}
=
r_{\rm PPO}
\exp(D_{\rm old}-D_\theta)
}
$$

这个等式非常重要。

它说明：

**ELBO Ratio 并不等于真实策略 Ratio，而是多了一个 ELBO Gap 修正项。**

只有当：

$$
D_\theta=D_{\rm old}
$$

时：

$$
r_{\rm ELBO}=r_{\rm PPO}
$$

## 6.2 VPSDE 下的 FPO Ratio

对于标准 VPSDE ELBO 目标：

$$
L_\theta
=
-\operatorname{ELBO}_\theta+C
$$

因此：

$$
\operatorname{ELBO}_\theta
=
-L_\theta+C
$$

代入：

$$
r_{\rm ELBO}
=
\exp
\left[
(-L_\theta+C)
-
(-L_{\rm old}+C)
\right]
$$

常数抵消：

$$
\boxed{
r_{\rm FPO}
=
\exp(L_{\rm old}-L_\theta)
}
$$

在这一设置下，$L_\theta$ 是与 ELBO 对应的加权去噪损失。

因此：

$$
\boxed{
\text{VPSDE：ELBO Ratio}
=
\exp(\text{新旧加权去噪损失之差})
}
$$

这是直接的变分推导。

## 6.3 Flow Matching 下的 FPO Ratio

对于 OT-CFM，情况有所不同。

我们得到的是：

$$
L_{\rm FM,\theta}
=
-\mathcal E_\theta+C
$$

其中：

$$
\mathcal E_\theta
=
\mathbb E_{t\sim p_w}
\mathbb E_{x_t\sim q(x_t|a)}
\left[
\operatorname{ELBO}_{\theta,t}(x_t)
\right]
$$

这里暂时假设权重已经按对应规范归一化。

构造 Expected ELBO Ratio：

$$
\boxed{
r_{\rm FPO}
=
\exp
\left(
\mathcal E_\theta-\mathcal E_{\rm old}
\right)
}
$$

代入：

$$
\mathcal E_\theta=-L_{\rm FM,\theta}+C
$$

得到：

$$
\begin{aligned}
r_{\rm FPO}
&=
\exp
\left[
-L_{\rm FM,\theta}+C
+L_{\rm FM,old}-C
\right]\\
&=
\exp
\left[
L_{\rm FM,old}-L_{\rm FM,\theta}
\right]
\end{aligned}
$$

因此：

$$
\boxed{
r_{\rm FPO}
=
\exp
\left(
L_{\rm FM,old}-L_{\rm FM,\theta}
\right)
}
$$

这就是 FPO 的核心公式。

但需要强调：

**这里的 Ratio 是 Expected ELBO 构造的代理比值，不是通过数学恒等变换得到的干净动作真实 likelihood ratio。**

因此，完整的逻辑应该是：

$$
\boxed{
\begin{aligned}
&\text{OT-CFM 速度 MSE}\\
&\Downarrow\\
&\text{等价于特定加权去噪 MSE}\\
&\Downarrow\\
&\text{单调权重对应 Expected ELBO}\\
&\Downarrow\\
&\text{用 Expected ELBO 构造代理 Ratio}\\
&\Downarrow\\
&r_{\rm FPO}
=
e^{L_{\rm old}-L_\theta}
\end{aligned}
}
$$

而不是简单地说：

$$
L_{\rm FM}=-\log\pi_\theta(a)
$$

后者并不成立。

---

# 七、Monte Carlo 估计与偏差

理论上的 FM Loss 为：

$$
L_{\rm FM,\theta}(a,s)
=
\mathbb E_{t,\epsilon}
\left[
\ell_\theta(t,\epsilon;a,s)
\right]
$$

实际使用 Monte Carlo 估计：

$$
\boxed{
\widehat L_\theta
=
\frac1N
\sum_{i=1}^{N}
\ell_\theta(t_i,\epsilon_i)
}
$$

因此：

$$
\boxed{
\widehat r_{\rm FPO}
=
\exp
\left[
\frac1N
\sum_{i=1}^{N}
\left(
\ell_{\rm old}^{(i)}
-
\ell_\theta^{(i)}
\right)
\right]
}
$$

## 7.1 为什么新旧模型使用相同噪声？

因为需要减少 Monte Carlo 方差。

如果使用不同的时间和噪声，即使：

$$
\theta=\theta_{\rm old}
$$

也可能得到：

$$
\widehat L_\theta\neq\widehat L_{\rm old}
$$

导致：

$$
\widehat r_{\rm FPO}\neq1
$$

因此，FPO 使用相同的：

$$
(t_i,\epsilon_i)
$$

计算新旧模型的损失。

## 7.2 Ratio 估计为什么有偏？

设：

$$
D_i=\ell_{\rm old}^{(i)}-\ell_\theta^{(i)}
$$

则：

$$
\widehat r=e^{\bar D}
$$

由于指数函数是凸函数，根据 Jensen 不等式：

$$
\boxed{
\mathbb E[\widehat r]
\geq
e^{\mathbb E[\bar D]}
=
r_{\rm FPO}
}
$$

因此，有限 Monte Carlo 样本的 Ratio 通常存在向上偏差。

增加 $N$ 可以降低损失差估计的方差。

但需要区分：

$$
\mathbb E[\nabla_\theta\widehat L_\theta]
=
\nabla_\theta L_\theta
$$

与：

$$
\mathbb E[\nabla_\theta\widehat r]
=
\nabla_\theta r
$$

后者一般不成立。

因此，**FM Loss 梯度无偏，不代表完整 FPO Ratio 梯度无偏。**

---

# 八、FPO 的 PPO 优化目标

FPO 将代理比值代入 PPO：

$$
\boxed{
J_{\rm FPO}
=
\mathbb E
\left[
\min
\left(
r_{\rm FPO}A,
\operatorname{clip}(r_{\rm FPO},1-\epsilon,1+\epsilon)A
\right)
\right]
}
$$

其中：

$$
r_{\rm FPO}
=
e^{L_{\rm old}-L_\theta}
$$

## 8.1 正优势样本

如果：

$$
A>0
$$

希望：

$$
r_{\rm FPO}\uparrow
$$

因此：

$$
L_\theta\downarrow
$$

即强化高奖励动作。

## 8.2 负优势样本

如果：

$$
A<0
$$

希望：

$$
r_{\rm FPO}\downarrow
$$

因此：

$$
L_\theta\uparrow
$$

即抑制低奖励动作。

所以：

$$
\boxed{
\begin{aligned}
A>0&:\quad L_{\rm FM}\downarrow\\
A<0&:\quad L_{\rm FM}\uparrow
\end{aligned}
}
$$

## 8.3 与奖励加权 Flow Matching 的关系

令：

$$
\Delta L=L_\theta-L_{\rm old}
$$

则：

$$
r_{\rm FPO}=e^{-\Delta L}
$$

当：

$$
|\Delta L|\ll1
$$

时：

$$
e^{-\Delta L}\approx1-\Delta L
$$

因此：

$$
r_{\rm FPO}A
\approx
A-A(L_\theta-L_{\rm old})
$$

对于固定 Rollout 数据，最大化这个目标在局部近似下等价于最小化：

$$
\boxed{
L_{\rm weighted}
=
\mathbb E[A L_{\rm FM,\theta}]
}
$$

因此，FPO 可以理解为：

**在优势加权 Flow Matching 回归的基础上，引入指数代理比值和 PPO Clipping。**

---

# 九、FPO 完整训练流程

FPO 使用 Actor-Critic 结构。

Actor：

$$
v_\theta(x_t,t,s)
$$

Critic：

$$
V_\phi(s)
$$

训练流程如下。

**Step 1：采集环境 Rollout。**

使用旧 Flow Policy：

$$
a\sim\pi_{\rm old}(\cdot|s)
$$

与环境交互，获得：

$$
(s,a,r,s')
$$

**Step 2：计算优势。**

论文主要使用 GAE：

$$
\delta_t
=
r_t+\gamma V_\phi(s_{t+1})-V_\phi(s_t)
$$

$$
\widehat A_t
=
\sum_{l\geq0}
(\gamma\lambda)^l\delta_{t+l}
$$

**Step 3：采样 FM 时间与噪声。**

对于每个动作：

$$
a
$$

采样：

$$
(t_i,\epsilon_i)
$$

构造：

$$
x_{t_i}
=
(1-t_i)a+t_i\epsilon_i
$$

**Step 4：计算旧策略 FM Loss。**

$$
L_{\rm old}
=
\frac1N
\sum_i
\left\|
v_{\rm old}(x_{t_i},t_i,s)
-
(\epsilon_i-a)
\right\|^2
$$

**Step 5：计算新策略 FM Loss。**

$$
L_\theta
=
\frac1N
\sum_i
\left\|
v_\theta(x_{t_i},t_i,s)
-
(\epsilon_i-a)
\right\|^2
$$

**Step 6：计算 FPO Ratio。**

$$
r_{\rm FPO}
=
e^{L_{\rm old}-L_\theta}
$$

**Step 7：使用 PPO Clipping 更新 Actor。**

$$
L_{\rm actor}
=
-
\mathbb E
\left[
\min
\left(
r_{\rm FPO}A,
\operatorname{clip}(r_{\rm FPO},1-\epsilon,1+\epsilon)A
\right)
\right]
$$

**Step 8：更新 Critic。**

通过价值函数回归更新：

$$
V_\phi
$$

然后重新采集 Rollout。

整个训练过程不需要保存 Flow ODE 的内部积分链，也不需要计算精确动作概率。

---

# 十、实验设计与结果

论文主要验证三个问题：

1. FPO 能否学习多模态动作分布？
2. FPO 能否在连续控制任务中优于 Gaussian PPO？
3. Flow Policy 的表达能力能否改善条件不足的机器人控制？

## 10.1 GridWorld

作者构建：

$$
25\times25
$$

的二维环境。

在部分状态下，智能体可以选择多个合理动作。

实验表明，FPO 能够学习多模态动作分布，并通过不同初始噪声生成不同的合理行为。

## 10.2 MuJoCo Playground

论文在 10 个连续控制任务中比较 FPO、Gaussian PPO 和 DPPO。

| 方法 | 平均奖励 |
|---|---:|
| Gaussian PPO | $667.8\pm66.0$ |
| DPPO | $652.5\pm83.7$ |
| FPO，MC=1 | $691.6\pm50.3$ |
| FPO，MC=4 | $731.2\pm58.2$ |
| **FPO，MC=8** | **$759.3\pm45.3$** |

实验说明：

- FPO 的平均性能优于对比方法。
- 增加 Monte Carlo 样本数通常能够改善性能。
- FPO 对 PPO Clipping 参数比较敏感。

## 10.3 Humanoid Control

| 条件输入 | Gaussian PPO | FPO |
|---|---:|---:|
| 全部关节目标 | 98.7% | 96.4% |
| Root + Hands | 46.5% | **70.6%** |
| 仅 Root | 29.8% | **54.3%** |

当条件充分时，FPO 不一定具有优势。

但在条件不足、动作存在多种合理解时，FPO 的表达能力更加明显。

这些结果支持了论文关于 Flow Policy 多模态建模优势的主要论点。[arXiv](https://arxiv.org/html/2507.21053?utm_source=chatgpt.com)

---

# 十一、FPO、ReinFlow 与 Diffusion-GRPO 的区别

| 维度 | FPO | ReinFlow | Diffusion-GRPO |
|---|---|---|---|
| 核心思想 | ELBO 代理概率比值 | 可学习噪声构造随机策略 | 随机去噪链 + 组内优势 |
| 是否需要随机去噪 | 否 | 是 | 通常是 |
| 是否保存去噪链 | 否 | 是 | 是 |
| 优化对象 | FM Loss Ratio | 高斯转移概率 | 去噪链转移概率 |
| 是否需要精确动作 likelihood | 否 | 否 | 否 |
| 是否直接优化真实概率比值 | 否 | 优化随机路径概率 | 优化随机路径概率 |
| 是否与求解器绑定 | 较弱 | 与随机采样构造有关 | 与采样链构造有关 |
| 主要问题 | ELBO Ratio 代理误差 | 训练与部署分布差异 | 路径概率与动作边缘概率不同 |

三者解决的是同一个问题：

$$
\text{生成模型难以直接计算动作概率}
$$

但选择了不同路线。

FPO：

$$
\boxed{
\text{用 ELBO 代理 likelihood}
}
$$

ReinFlow：

$$
\boxed{
\text{将生成过程随机化，使转移概率可计算}
}
$$

Diffusion-GRPO：

$$
\boxed{
\text{利用随机去噪链计算策略更新}
}
$$

---

# 十二、FPO 的理论局限

## 12.1 ELBO Ratio 不等于真实概率比值

$$
r_{\rm FPO}
=
r_{\rm PPO}
e^{D_{\rm old}-D_\theta}
$$

因此，即使：

$$
r_{\rm FPO}\approx1
$$

也不能保证：

$$
r_{\rm PPO}\approx1
$$

所以 FPO Clipping 不构成严格的真实策略 KL 约束。

## 12.2 Flow Matching 对应 Expected ELBO，而不是直接对应干净动作 ELBO

这是本文最需要强调的理论区别。

对于标准 VPSDE：

$$
\boxed{
L_{\rm ELBO}
=
-\operatorname{ELBO}_{\rm clean}+C
}
$$

对于普通 OT-CFM：

$$
\boxed{
L_{\rm FM}
\longleftrightarrow
-\operatorname{ExpectedELBO}_{\rm noisy}
}
$$

因此，FPO 将普通 FM Loss 用于构造动作概率代理时，仍然存在理论上的近似关系。

## 12.3 Monte Carlo Ratio 存在偏差

$$
\mathbb E[e^{\widehat{\Delta L}}]
\neq
e^{\mathbb E[\widehat{\Delta L}]}
$$

所以有限采样的 FPO Ratio 一般不是无偏估计。

## 12.4 预训练模型后训练稳定性尚未充分验证

论文主要实验采用从零开始的在线 RL。

对于大规模预训练生成模型，论文的图像微调实验观察到了分布漂移和生成质量下降。

因此，不能仅凭机器人控制实验就认定 FPO 一定适合稳定的预训练模型后训练。

---

# 十三、最终总结

FPO 的理论可以分成三层。

### 第一层：扩散模型为什么能够使用 ELBO？

从 DDPM 的变分下界出发：

$$
\boxed{
\log p_\theta(x_0)
\geq
\operatorname{ELBO}_\theta(x_0)
}
$$

将负 ELBO 分解为反向高斯 KL：

$$
\boxed{
-\operatorname{ELBO}
=
\sum_kD_{\rm KL}(q_{\rm posterior}\|p_\theta)
+
\text{边界项}
}
$$

由于反向过程采用高斯参数化：

$$
\boxed{
D_{\rm KL}
\longrightarrow
w_k\|\epsilon_\theta-\epsilon\|^2
}
$$

连续时间 VPSDE 下：

$$
\boxed{
-\operatorname{ELBO}_\theta
=
\frac12
\mathbb E
\left[
-\dot\lambda
\|\epsilon_\theta-\epsilon\|^2
\right]+C
}
$$

因此，加权去噪 MSE 与 ELBO 之间存在严格的变分联系。

### 第二层：Flow Matching 为什么也能联系到 ELBO？

OT-CFM 路径：

$$
x_t=(1-t)x_0+t\epsilon
$$

满足：

$$
\boxed{
\|\epsilon_\theta-\epsilon\|^2
=
(1-t)^2\|v_\theta-u_t\|^2
}
$$

因此，普通 FM Loss 可以表示成：

$$
\boxed{
L_{\rm FM}
=
\frac12
\mathbb E
\left[
w(\lambda)
(-\dot\lambda)
\|\epsilon_\theta-\epsilon\|^2
\right]
}
$$

其中：

$$
\boxed{
w(\lambda)=e^{-\lambda/2}
}
$$

由于这个权重满足单调性条件，可以利用 Kingma–Gao 的结果，将其解释为噪声增强数据上的 Expected ELBO 优化。

因此：

$$
\boxed{
L_{\rm FM}
\longleftrightarrow
-\operatorname{ExpectedELBO}
}
$$

这一联系需要适当处理噪声端点及权重归一化。

### 第三层：FPO 如何构造 PPO Ratio？

使用 ELBO 代理真实对数概率：

$$
\boxed{
r_{\rm ELBO}
=
\exp(
\operatorname{ELBO}_\theta-
\operatorname{ELBO}_{\rm old}
)
}
$$

再利用去噪损失与 ELBO 的关系：

$$
\boxed{
r_{\rm FPO}
=
\exp(L_{\rm old}-L_\theta)
}
$$

最终：

$$
\boxed{
J_{\rm FPO}
=
\mathbb E
\left[
\min
\left(
r_{\rm FPO}A,
\operatorname{clip}(r_{\rm FPO},1-\epsilon,1+\epsilon)A
\right)
\right]
}
$$

**整篇论文最重要的理论结论是：**

> FPO 并不是直接证明 Flow Matching Loss 等于负对数概率，而是先利用扩散模型的变分 ELBO 理论，再通过加权去噪目标建立 Flow Matching 与 Expected ELBO 的联系，最后使用 ELBO 的相对变化构造 PPO 风格的策略概率代理比值。

这也是为什么 FPO 可以不计算精确 likelihood、不构造随机去噪 MDP，就直接对 Flow Policy 使用在线强化学习。

**但从严格数学意义上说，FPO 优化的是 ELBO 构造的代理目标，而不是完全等价的真实策略梯度目标。**